# 01 — Exploratory Data Analysis (UAVVaste)

**Project:** CNN-Based Aerial Litter Detection for Sustainable Trail and Environmental Cleanup  
**Module:** ST7088CEM Artificial Neural Networks

This notebook builds leakage-free image-level splits and produces the dataset
statistics and figures used in the report, from the UAVVaste COCO annotation
file alone — the 2.9 GB image archive is only needed later, for tile-crop
generation and detector training.

## 1. Environment setup

On **Google Colab / Kaggle**: uncomment the clone + install lines.
Locally: run from the repository's `notebooks/` folder with requirements installed.

In [ ]:
# --- On Kaggle (Internet enabled in notebook settings), uncomment: ---
# !git clone https://github.com/Sajan491/STW7088CEM-ANN-Assignment.git
# %cd STW7088CEM-ANN-Assignment
# %pip install -q -r requirements.txt
#
# The uploaded dataset mounts read-only under /kaggle/input/<dataset-slug>/.
# Check its structure, then link images + annotations into ./data:
# !ls /kaggle/input
# import os; os.makedirs('data', exist_ok=True)
# !ln -s /kaggle/input/<dataset-slug>/images data/images
# !ln -s /kaggle/input/<dataset-slug>/annotations data/annotations

# --- On Colab, uncomment instead: ---
# !git clone https://github.com/Sajan491/STW7088CEM-ANN-Assignment.git
# %cd STW7088CEM-ANN-Assignment
# %pip install -q -r requirements.txt
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r /content/drive/MyDrive/UAVVaste/data ./data

import os, sys, platform
from pathlib import Path

# Make the repo root the working directory whether we start in notebooks/ or root
if Path.cwd().name == 'notebooks':
    os.chdir('..')
sys.path.insert(0, str(Path.cwd()))

print('Working directory:', Path.cwd())
print('Python:', sys.version)
print('Machine:', platform.node(), '|', platform.platform())

## 2. Dataset files

The dataset is downloaded manually and is never committed to the repo:

- Annotations: [annotations.json](https://raw.githubusercontent.com/PUTvision/UAVVaste/main/annotations/annotations.json) → `data/annotations/annotations.json`
- Images: [UAVVasteDataset.zip on Zenodo](https://zenodo.org/records/8214061) (~2.9 GB) → extracted flat into `data/images/`

This cell verifies the annotation file is in place (images are optional here).

In [ ]:
ann = Path('data/annotations/annotations.json')
assert ann.exists(), (
    'annotations.json not found — download it manually (see links above) '
    'and place it at data/annotations/annotations.json'
)
print(f'annotations: {ann} ({ann.stat().st_size / 1e6:.1f} MB)')

images = sorted(Path('data/images').glob('*.[jJpP]*'))
print(f'images: {len(images)} found in data/images/ (not required for this notebook)')

## 3. Leakage-free image-level splits

Images (not tiles) are partitioned 70/15/15 with a fixed seed, so every tile
later inherits its source image's split and no image ever crosses splits.

In [ ]:
!python -m src.data.splits

## 4. Dataset statistics, figures and tables

Produces the object-size distribution, annotations-per-image histogram and the
tile class-balance figure (tile labels are computed geometrically from the
annotations, so no images are required).

In [ ]:
!python -m src.data.eda

## 5. Figures

In [ ]:
from IPython.display import Image, display

for name in ['object_size_distribution', 'annotations_per_image', 'tile_class_balance']:
    display(Image(f'results/figures/{name}.png', width=800))

## 6. Tables

In [ ]:
import pandas as pd

print('Dataset summary:')
display(pd.read_csv('results/tables/dataset_summary.csv', index_col=0))
print('Tile class balance:')
display(pd.read_csv('results/tables/tile_class_balance.csv'))

## 7. Observations

- **Small-object problem confirmed:** ~99% of litter bounding boxes occupy less
  than 1% of the image area (median ≈ 0.07%). Standard 640 px detector input
  will shrink most objects to a few pixels — motivating the tiling approach
  (Task 1) and SAHI sliced inference + higher resolution (Task 2).
- **Every image contains litter** (no empty images), with a right-skewed
  distribution of ~4.8 annotations per image (max 65).
- **Tile imbalance:** at 512 px tiles only ~15% are positive, so the tile
  classifier uses a weighted loss / balanced sampling.
- Splits are disjoint at image level (verified programmatically and by unit
  tests in `tests/`).